# Test de LDA pour l'analyse de topic

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [3]:
df=pd.read_csv("corpus_zola_nettoye.csv") # Chargement du DataFrame nettoyé à partir du fichier CSV

In [20]:
def segmenter_texte(text, taille=150): # On segmente le texte en segments de 200 mots
    mots = text.split() # On divise le texte en mots
    segments = [] # Liste pour stocker les segments de texte
    for i in range(0, len(mots), taille): # On boucle sur les mots par pas de "taille" (300 mots)
        segment = " ".join(mots[i:i+taille]) # On crée un segment en joignant les mots du segment
        if len(segment.split()) >= 100:  # On vérifie que le segment contient au moins 100 mots pour éviter les segments trop courts
            segments.append(segment) # On ajoute le segment à la liste des segments
    return segments # On retourne la liste des segments de texte

lignes = [] # Liste pour stocker les segments de texte avec leurs titres et IDs

for _, row in df.iterrows():
    titre = row["nom_fichier"] # On utilise le nom du fichier comme titre
    texte = row["texte_nettoye"] # On segmente le texte en segments de 300 mots
    segments = segmenter_texte(texte) # On ajoute chaque segment à la liste avec son titre et son ID
    
    for j, seg in enumerate(segments):
        
        
        lignes.append({ 
            "titre": titre, # Titre du texte (nom du fichier)
            "segment_id": j, # ID du segment (index du segment dans le texte) 
            "texte_segment": seg # Contenu du segment de texte
        })

df_segments = pd.DataFrame(lignes) # Création d'un DataFrame à partir de la liste de segments

df_segments

,titre,segment_id,texte_segment
0,Carnets d'enquetes - Emile Zola_clean.txt,0,bruit étourdissement impossibilité place fixit...
1,Carnets d'enquetes - Emile Zola_clean.txt,1,travers boire champagne déformer nom fâcher ex...
2,Carnets d'enquetes - Emile Zola_clean.txt,2,mouchoir accuser monde voler redemander table ...
3,Carnets d'enquetes - Emile Zola_clean.txt,3,geste voluptueux envoyer amoureux salve baiser...
4,Carnets d'enquetes - Emile Zola_clean.txt,4,héros grec zola visite compagnie prendre note ...
...,...,...,...
11496,1894_1_Lourdes._clean.txt,441,chair renoncer femme sentir père emporter sacr...
11497,1894_1_Lourdes._clean.txt,442,forcer porte mystère mot sonner crâne absorber...
11498,1894_1_Lourdes._clean.txt,443,vérité conquis religion appétit mort vivre mou...
11499,1894_1_Lourdes._clean.txt,444,monde pèlerin réclamer larme égalité santé par...


In [21]:
df_segments["nb_tokens_nettoyes"] = df_segments["texte_segment"].apply(lambda x: len(x.split())) # Calcul du nombre de tokens nettoyés pour chaque segment
df_segments["nb_tokens_nettoyes"].describe() # Affichage des statistiques descriptives du nombre de tokens nettoyés par segment (moyenne, écart-type, min, max, etc.)

count    11501.000000
mean       149.981741
std          0.781980
min        104.000000
25%        150.000000
50%        150.000000
75%        150.000000
max        150.000000
Name: nb_tokens_nettoyes, dtype: float64

In [22]:
vectorizer = CountVectorizer(
    max_df=0.8, # Ignorer les termes qui apparaissent dans plus de 80% des segments
    min_df=5, # Ignorer les termes qui apparaissent dans moins de 5 segments
    #max_features=5000 # Limiter le nombre de termes à 5000 pour réduire la dimensionnalité de la matrice de caractéristiques
)

X = vectorizer.fit_transform(df_segments["texte_segment"])  
#Appliquer le TF-IDF vectorizer sur les segments de texte nettoyés pour obtenir une matrice de caractéristiques (termes pondérés par leur importance dans les segments).

X.shape

(11501, 11954)

In [23]:
lda = LatentDirichletAllocation(
    n_components=10,
    learning_method="batch",
    random_state=42,
    max_iter=30,
    doc_topic_prior=0.1,
    topic_word_prior=0.1
)


W = lda.fit_transform(X)
H = lda.components_


feature_names = vectorizer.get_feature_names_out()

def afficher_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topic {topic_idx+1} : {' | '.join(top_words)}")

afficher_topics(lda, feature_names, 10)

Topic 1 : travail | ouvrier | machine | train | rue | heure | maison | jour | chef | grand
Topic 2 : petit | manger | rue | grand | gros | coup | table | mettre | vieux | eau
Topic 3 : grand | soleil | petit | arbre | ciel | eau | route | heure | haut | noir
Topic 4 : femme | grand | petit | blanc | jeune | oeil | air | rire | monsieur | main
Topic 5 : fille | enfant | femme | petit | père | mère | grand | bon | franc | jour
Topic 6 : homme | devoir | affaire | grand | trouver | frère | mettre | donner | coup | jour
Topic 7 : main | oeil | mort | homme | sentir | prendre | coup | rester | bras | entendre
Topic 8 : abbé | prêtre | église | saint | grand | curé | pape | cardinal | palais | salle
Topic 9 : vie | grand | monde | aimer | enfant | jour | vivre | amour | cœur | bonheur
Topic 10 : monsieur | heure | bon | regarder | prendre | demander | répondre | petit | air | femme
